## Task 0: Classical Skeleton

A 3x3 grid represented as a list of 9 cells, each either empty, 'X', or 'O'

A game loop that alternates Player 1 (X) and Player 2 (O)

In [46]:
# quantum_ttt.py
#The board starts as ['1', '2',...,'9'] and as players make moves, the numbers are replaced with 'X' or 'O'.
# A number on the board represents an available move, while 'X' or 'O' represents a move made by a player.
board = [str(i) for i in range(1, 10)] #['1', '2',.....'9']

def display_board(board):
    print()
    print(' ' + board[0] + ' | ' + board[1] + ' | ' + board[2])
    print('---+---+---')
    print(' ' + board[3] + ' | ' + board[4] + ' | ' + board[5])
    print('---+---+---')
    print(' ' + board[6] + ' | ' + board[7] + ' | ' + board[8])
    print() #Print the 3*3 grid with the current state of the board.


def game_loop():
    players = ["X", "O"] #Two players, 'X' and 'O'.
    
    for turn in range(9): #Maximum of 9 turns in a game of tic-tac-toe.
        display_board(board) #Display the current state of the board at the start of each turn.

        player = players[turn %2] #Alternate between player 'X' and 'O' based on the turn number.

        while True:
            choice = input(f"Player{player}, pick a square(1-9)")

            # if the choice is still in the board, the cell is empty and the input is valid.
            # if it's been replaced by 'X' od 'O', or doesn't exist, it fails

            if choice in board:
                board[int(choice) - 1] = player #replace the number with X or O
                break #valid move made, exit the while loop
            print("Invalid move. Try again.") #Prompt the player to try again if the move is invalid.


        # Show the final board state after all turns
        display_board(board)

    game_loop()



## Task 1: The Quantum Backend

This is a quantum circuit with 9 qubits and 9 classical bits for eventual measurement.

Also write a helper that links the python board to the circuit


In [45]:
from qiskit import QuantumCircuit

""" I need 9 qubits for each square on the board, think of each qubit as the quantum state of that square"""


def init_quantum_backend():
    global qc
    qc = QuantumCircuit(9, 9)
    qc.draw('mpl')
    # This function will initialize the quantum backend, which is necessary for running quantum circuits.
    print("Quantum backend initialized.")

def square_to_qubit(square):
    # This function will map a square number (1-9) to a corresponding qubit index (0-8).
    return int(square) - 1
# Call this once at the start of the game
init_quantum_backend()

# Test the helper function
for square in range(1, 10):
    qubit_index = square_to_qubit(square)
    print(f"Square {square} maps to qubit index {qubit_index}.")




Quantum backend initialized.
Square 1 maps to qubit index 0.
Square 2 maps to qubit index 1.
Square 3 maps to qubit index 2.
Square 4 maps to qubit index 3.
Square 5 maps to qubit index 4.
Square 6 maps to qubit index 5.
Square 7 maps to qubit index 6.
Square 8 maps to qubit index 7.
Square 9 maps to qubit index 8.


## Task 2: Implementing "Spooky" Moves
 In Quantum Tic-Tac-Toe a player picks two squares — their mark exists in both at the same time (superposition). Nobody knows which one it will actually land on until the state collapses later.

 To represent this in Qiskit:

A Hadamard gate on a qubit puts it into superposition — it becomes both 0 and 1 at the same time, representing the move being "uncertain"

A CNOT gate entangles two qubits together — meaning the two squares the player picked are now linked. Whatever happens to one affects the other

In [47]:
superposition_moves = []

#counts moves made, used to label marks e.g X1, X2, O1, O2, etc.
move_counter = 0

def get_two_squares(board):
    # Ask the player to pick two squares for their move
    squares = []
    for i in ["first", "second"]:
        while True:
            try:
                sq = int(input(f" Pick your {i} square (1-9): "))
                if 1 <= sq <= 9 and str(sq) in board:
                    squares.append(str(sq))
                    break
                else:
                    print("Invalid choice. Try again.")
            except ValueError:
                print("Please enter a valid number.")
    return int(squares[0]), int(squares[1]) # return both as integers for easier processing later


# ========================
# TASK 2: FIX apply_quantum_gates()
# ========================

def apply_quantum_gates(q1, q2):
    """
    Apply quantum gates to represent a superposition move.
    
    We create a fresh mini circuit for each move to avoid
    duplicate qubit errors from reusing the same global circuit.
    
    Step 1 — Hadamard gate on q1:
        Puts qubit q1 into superposition.
        It becomes both 0 and 1 at the same time.
        This represents the move being uncertain.
    
    Step 2 — CNOT gate on q1 (control) and q2 (target):
        Entangles the two qubits together.
        If q1 collapses to 0, q2 stays 0.
        If q1 collapses to 1, q2 flips to 1.
    """
    qc.h(q1) # Step 1: Put q1 into superposition
    qc.cx(q1, q2) # Step 2: Entangle q1 and q2 with a CNOT gate

def make_spooky_move(board, player):
    """A spooky move lets the player place their mark in TWO squares at once.
    The mark exists in both squares simultaneously until collapse in Task 3 """
    global move_counter
    move_counter += 1

    # Get two squares from the player
    sq1, sq2 = get_two_squares(board)

    # Convert squares (1-9) to qubit indices (0-8) using the helper
    q1, q2 = square_to_qubit(sq1), square_to_qubit(sq2)

    #Apply Hadamard + CNOT to entangle the two qubits
    apply_quantum_gates(q1, q2)

    #label for this move e.g X1, X2, O1, O2, etc.
    label = f"{player}{move_counter}"

    # update the board with the label for both squares
    board[sq1-1] = label
    board[sq2-1] = label

    #Store the move for the superposition collapse in Task 3
    superposition_moves.append({'label': label, 'squares': (sq1, sq2)})

    print(f" {label} placed in squares {sq1} and {sq2}.\n")

## Task 3: State Collapse



Collapse is the moment reality is forced to choose- each superposition mark lands on exactly one of its two squares



What's happening here:

detect_loop() — builds a graph from all superposition moves and checks for a cycle. A cycle means collapse must happen

measure_qubits() — adds measurement gates to the circuit, runs it on AerSimulator for 1 shot, returns the binary result string

collapse_board() — uses the measurement result to decide where each mark permanently lands, updates the board, frees losing squares

In [48]:

from qiskit_aer import AerSimulator


def detect_loop(superposition_moves):
    """
    Check if the superposition moves form a closed loop.
    
    A loop means a cycle exists where squares are shared between moves.
    Example of a loop:
        X1 on squares (1, 5)
        O1 on squares (5, 9)
        X2 on squares (9, 1)
        Square 1 -> 5 -> 9 -> 1 forms a cycle — collapse must happen!
    
    We detect this by building a graph where:
        - Each square is a node
        - Each superposition move is an edge between two squares
    A loop exists when any node is visited twice.
    """
    # Build a simple adjacency list from all superposition moves
    # Example: {1: [5], 5: [1, 9], 9: [5, 1]}
    graph = {}
    for move in superposition_moves:
        sq1, sq2 = move['squares']
        graph.setdefault(sq1, []).append(sq2)
        graph.setdefault(sq2, []).append(sq1)

    # Walk the graph to detect a cycle using simple visited tracking
    visited = set()

    def has_cycle(node, parent):
        """Recursively walk the graph looking for a cycle."""
        visited.add(node)
        for neighbor in graph.get(node, []):
            if neighbor not in visited:
                if has_cycle(neighbor, node):
                    return True
            # If we reach a visited node that isn't our parent, it's a cycle
            elif neighbor != parent:
                return True
        return False

    # Check every node in case the graph is disconnected
    for node in graph:
        if node not in visited:
            if has_cycle(node, -1):
                return True  # loop found — trigger collapse

    return False  # no loop yet


def measure_qubits(involved_qubits):
    """
    Measure only the qubits involved in the collapse.
    
    Steps:
    1. Add measurement gates to the involved qubits in the circuit
    2. Run the circuit on AerSimulator for exactly 1 shot
    3. Return the measurement result as a string of bits e.g. '001101001'
    
    1 shot means the simulator picks one definitive outcome —
    this is the quantum randomness that decides where marks land.
    """
    # Create a copy of the circuit to avoid modifying the original
    collapse_circuit = qc.copy()

    # Add measurement gates only to the qubits involved in this collapse
    for q in involved_qubits:
        # measure(qubit_index, classical_bit_index)
        # Result gets stored in the classical bit at the same index
        collapse_circuit.measure(q, q)

    # Run the circuit on AerSimulator for 1 shot
    simulator = AerSimulator()
    job = simulator.run(collapse_circuit, shots=1)
    result = job.result()

    # Get the measurement result — a binary string e.g. '001101001'
    # Each character represents the collapsed state of one classical bit
    counts = result.get_counts()
    measured_bits = list(counts.keys())[0]  # only 1 shot so only 1 result

    # Qiskit returns bits in reverse order (rightmost = qubit 0)
    # Reverse it so index 0 = qubit 0
    return measured_bits[::-1]


def collapse_board(board, superposition_moves):
    """
    Collapse all superposition moves into definitive classical marks.
    
    For each superposition move:
    - Look at the measurement result for both its qubits
    - If qubit of sq1 measured 1 — the mark lands on sq1
    - If qubit of sq1 measured 0 — the mark lands on sq2
    - The other square is freed up (reset to its number)
    
    The board updates permanently from labels (X1, O1) to marks (X, O).
    """
    # Collect all qubits involved in superposition moves
    involved_qubits = set()
    for move in superposition_moves:
        sq1, sq2 = move['squares']
        involved_qubits.add(square_to_qubit(sq1))
        involved_qubits.add(square_to_qubit(sq2))

    # Measure those qubits and get the result
    measured_bits = measure_qubits(involved_qubits)

    print("\n--- COLLAPSE TRIGGERED ---")
    print(f"Measurement result: {measured_bits}\n")

    # Resolve each superposition move based on measurement
    for move in superposition_moves:
        sq1, sq2 = move['squares']
        player = move['label'][0]  # 'X' or 'O' from label e.g. 'X1'

        q1 = square_to_qubit(sq1)  # qubit index for sq1

        # If qubit q1 measured as 1 — mark lands on sq1
        # If qubit q1 measured as 0 — mark lands on sq2
        if measured_bits[q1] == '1':
            winning_square = sq1
            losing_square = sq2
        else:
            winning_square = sq2
            losing_square = sq1

        # Place the permanent mark on the winning square
        board[winning_square - 1] = player

        # Free up the losing square — reset it to its number
        board[losing_square - 1] = str(losing_square)

        print(f"  {move['label']} collapses to square {winning_square} for Player {player}")

    # Clear all superposition moves after collapse
    superposition_moves.clear()
    print("\nBoard after collapse:")

## Task 4: Win Conditions

In [ ]:
# All possible 3-in-a-row combinations on a 3x3 board
# Each tuple is a set of 3 board indices (0-indexed)
WIN_CONDITIONS = [
    (0, 1, 2),  # top row
    (3, 4, 5),  # middle row
    (6, 7, 8),  # bottom row
    (0, 3, 6),  # left column
    (1, 4, 7),  # middle column
    (2, 5, 8),  # right column
    (0, 4, 8),  # diagonal top-left to bottom-right
    (2, 4, 6),  # diagonal top-right to bottom-left
]


def check_winner(board):
    """
    Check the board for 3-in-a-row after a collapse.
    
    Only checks for classical marks (X or O) — not superposition labels.
    Returns:
        'X'  — if X wins
        'O'  — if O wins
        'XO' — if both win simultaneously (quantum edge case)
        None — if no winner yet
    """
    x_wins = False
    o_wins = False

    for combo in WIN_CONDITIONS:
        # Get the three cells for this combination
        a, b, c = board[combo[0]], board[combo[1]], board[combo[2]]

        # Check if all three cells are the same classical mark
        if a == b == c == 'X':
            x_wins = True
        if a == b == c == 'O':
            o_wins = True

    # Handle all possible outcomes
    if x_wins and o_wins:
        return 'XO'   # both win simultaneously — quantum edge case
    if x_wins:
        return 'X'
    if o_wins:
        return 'O'
    return None       # no winner yet


def display_scores(scores):
    """Display the current scores for both players."""
    print(f"\n  Scores — X: {scores['X']}  |  O: {scores['O']}\n")


def handle_winner(winner, scores):
    """
    Award points based on the winner.
    
    Classical win — one player gets 1 full point.
    Quantum edge case — both players get 0.5 points each.
    This handles the situation where a single collapse gives
    both players 3-in-a-row at the same time.
    """
    if winner == 'XO':
        # Both players get 3-in-a-row simultaneously
        # Quantum rules — award 0.5 points each
        scores['X'] += 0.5
        scores['O'] += 0.5
        print("  Both players got 3-in-a-row simultaneously!")
        print("  Quantum rules apply — each player gets 0.5 points.")
    else:
        # One player wins cleanly
        scores[winner] += 1
        print(f"  Player {winner} wins this round!")

    display_scores(scores)


def play_game():
    """
    The full polished game loop tying all tasks together.
    
    Flow:
    1. Initialize quantum backend (Task 1)
    2. Players take turns making superposition moves (Task 2)
    3. After every move, check if a loop has formed (Task 3)
    4. If loop detected — collapse the board (Task 3)
    5. After collapse — check for a winner (Task 4)
    6. Repeat until the board is full or a winner is found
    """
    # Fresh board and state for a new game
    board = [str(i) for i in range(1, 10)]
    players = ['X', 'O']
    scores = {'X': 0, 'O': 0}

    # Reset superposition tracking
    superposition_moves.clear()

    # Reset move counter
    global move_counter
    move_counter = 0

    # Initialize the quantum circuit (Task 1)
    init_quantum_backend()

    print("=== Quantum Tic-Tac-Toe ===\n")
    print("Rules:")
    print("  - Each turn you pick TWO squares for your mark")
    print("  - Your mark exists in both squares simultaneously")
    print("  - When moves form a loop, the board collapses")
    print("  - After collapse, 3-in-a-row wins!\n")

    for turn in range(9):
        display_board(board)
        player = players[turn % 2]

        # Player makes a superposition move (Task 2)
        make_spooky_move(board, player)

        # Check if superposition moves form a closed loop (Task 3)
        if detect_loop(superposition_moves):
            # Collapse the board — resolve all superposition marks (Task 3)
            collapse_board(board, superposition_moves)
            display_board(board)

            # Check for a winner after collapse (Task 4)
            winner = check_winner(board)
            if winner:
                handle_winner(winner, scores)
                break  # game over

    else:
        # All 9 turns used with no winner — it's a draw
        display_board(board)
        print("  No winner — it's a draw!")
        display_scores(scores)

    # Ask players if they want to play again
    again = input("\nPlay again? (y/n): ")
    if again.lower() == 'y':
        play_game()  # restart the game recursively
    else:
        print("\nThanks for playing Quantum Tic-Tac-Toe!")
        print(f"Final scores — X: {scores['X']}  |  O: {scores['O']}")


# Start the game
play_game()

Quantum backend initialized.
=== Quantum Tic-Tac-Toe ===

Rules:
  - Each turn you pick TWO squares for your mark
  - Your mark exists in both squares simultaneously
  - When moves form a loop, the board collapses
  - After collapse, 3-in-a-row wins!


 1 | 2 | 3
---+---+---
 4 | 5 | 6
---+---+---
 7 | 8 | 9

Please enter a valid number.
Please enter a valid number.
